# Data Exploration Notebook

This notebook helps profile the **raw datar** and define the **transformations**.


In [ ]:
%pip install pandas

import pandas as pd
import os

## 1) Select raw data

In [ ]:
folder_path = "../data/raw/"
raw_path = os.listdir(folder_path)
raw_data_path = ""
last_ingestion = 0

for filename in raw_path:
	if "jobs_" in filename and filename.endswith(".jsonl"):
		date_part  = filename.split("jobs_")[-1].split(".jsonl")[0]
		max_date = int(date_part.replace("_", ""))
		if max_date > last_ingestion:
		 	raw_data_path = filename

if not raw_data_path:
	raise ValueError("Raw data path not found.")
else:
	final_path = folder_path + raw_data_path
	df = pd.read_json(final_path, lines=True)
	# Initial conferences
	print(f"Rows count: {len(df):,}")
	print(f"Columns count: {len(df.columns)}")
	print(f"Columns: {df.columns}")


In [ ]:
# Transform 'raw' column dict in DataFrame
df_raw_columns = pd.json_normalize(df['raw'])
df = pd.concat([df, df_raw_columns], axis=1)
df = df.drop(columns=['raw'])

df.columns


## 2) Basic schema profiling

In [ ]:
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null": [int(df[c].notna().sum()) for c in df.columns],
    "nulls": [int(df[c].isna().sum()) for c in df.columns],
    "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(["null_pct", "column"], ascending=[False, True])

schema_df


In [ ]:
# Check the need to be normalized in Silver
key_dims = [c for c in df.columns]
for col in key_dims:
    print(f"\n=== {col} ===")
    display(
        df[col]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
        .value_counts(dropna=False)
        .head(10)
        .to_frame("count")
    )


## 3) Duplicate and key profiling

In [ ]:
if "id" in df.columns:
    dup_count = int(df.duplicated(subset=["id", "source"]).sum())
    print("Duplicated id per source:", dup_count)

    if dup_count:
        display(df[df.duplicated(subset=["id", "source"], keep=False)].sort_values("id").head(10))
else:
    print("Column 'id' not found.")
